# Lab 8 - Prove one improvement (browser only)

Anecdote is not proof. Here you baseline a query, make one change, clear the
cache, re-measure, and record the result - all in this notebook, no DAX Studio.

We use **semantic-link** to trace the query and **semantic-link-labs** to clear
the cache so each run is fair.

## 1. Install the helpers
The `%pip` line takes a few seconds the first time and needs only a browser. It **restarts the Python session**, so it has to run before you set anything else.

In [ ]:
%pip install semantic-link-labs -q

## 2. Point at your model

In [ ]:
# Resolves your own workspace, so there is nothing to hand-edit there.
import sempy.fabric as fabric

WORKSPACE = fabric.resolve_workspace_name()

# The model and its two queries are set TOGETHER on purpose. A query only works
# against a model that actually has the measures it names.
#
# MODULE 8 (default) - two variants, same answer, different cost. [Total Sales]
# sums the stored SalesAmount column, [Slow Sales (SE)] recomputes
# Quantity * UnitPrice * (1 - Discount) across all 3M rows.
MODEL     = "07 Slow Visual Triage"
SLOW_DAX  = """EVALUATE SUMMARIZECOLUMNS ( 'Date'[MonthYear], "Sales", [Slow Sales (SE)] )"""
TUNED_DAX = """EVALUATE SUMMARIZECOLUMNS ( 'Date'[MonthYear], "Sales", [Total Sales] )"""

# MODULE 4 (aggregations) - the same query before and after YOU change the model.
# Swap these in, then use Option A in section 4 instead of Option B.
# MODEL     = "04 Scaling"
# SLOW_DAX  = """EVALUATE SUMMARIZECOLUMNS ( 'Date'[MonthYear], "Sales", [Total Sales] )"""
# TUNED_DAX = None

print(f"workspace : {WORKSPACE}")
print(f"model     : {MODEL}")

In [ ]:
import sempy_labs as labs

# Server Timings, in a notebook. semantic-link (sempy) is already installed in
# Fabric, so there is nothing to install for this cell.
import sempy.fabric as fabric
import pandas as pd
import time

# The events we want. QueryEnd = the whole query; VertiPaqSEQueryEnd = storage
# engine (SE) scans. Formula engine (FE) time is the remainder.
EVENTS = {
    "QueryEnd":                  ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "VertiPaqSEQueryEnd":        ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "VertiPaqSEQueryCacheMatch": ["EventClass", "TextData"],
}

def _col(df, name):
    """Find a column ignoring spaces/case (sempy names vary by version)."""
    key = name.replace(" ", "").lower()
    for c in df.columns:
        if c.replace(" ", "").lower() == key:
            return c
    return None

def measure(dax, label, settle=5, show_raw=False):
    """Run a DAX query, trace it, and return a total / SE / FE breakdown in ms."""
    with fabric.create_trace_connection(dataset=MODEL, workspace=WORKSPACE) as tc:
        with tc.create_trace(EVENTS, "Perf trace") as tr:
            tr.start()
            fabric.evaluate_dax(dataset=MODEL, dax_string=dax, workspace=WORKSPACE)
            time.sleep(settle)                 # let the trace events flush
            logs = tr.stop()

    if show_raw or _col(logs, "EventClass") is None:
        display(logs)                          # fall back to the raw trace

    ec, dur = _col(logs, "EventClass"), _col(logs, "Duration")
    total = float(logs.loc[logs[ec] == "QueryEnd", dur].max() or 0)
    se    = float(logs.loc[logs[ec] == "VertiPaqSEQueryEnd", dur].sum() or 0)
    fe    = max(total - se, 0.0)
    bound = "SE-bound" if total and se / total >= 0.5 else "FE-bound"
    print(f"{label:<22}  total {total:>7.0f} ms   SE {se:>7.0f} ms   FE {fe:>7.0f} ms   -> {bound}")
    return {"Query": label, "Total ms": total, "SE ms": se, "FE ms": fe, "Bound": bound}

## 3. Benchmark: baseline -> change -> re-measure

The rule: **clear the cache, change ONE thing, re-measure.** The helper below
clears the cache before each run so warm caches cannot flatter your numbers.

Clearing the cache drops cached query *results*, so every run does the work
again. On Direct Lake it does **not** evict resident columns. That is why the
warm-up matters: the two queries read *different* columns, so without it
whichever ran first would pay the one-off cost of paging its columns in and the
other would look better than it is. Warm both, then compare compute to compute.

In [ ]:
def benchmark(dax, label):
    labs.clear_cache(dataset=MODEL, workspace=WORKSPACE)   # drops cached results, not columns
    return measure(dax, label)

# Warm-up: run both once and throw the numbers away, so every column either
# query needs is already resident before anything is measured.
for _dax in (SLOW_DAX, TUNED_DAX):
    if _dax:
        fabric.evaluate_dax(dataset=MODEL, dax_string=_dax, workspace=WORKSPACE)
print("warm-up done, columns resident\n")

baseline = benchmark(SLOW_DAX, "Baseline")

## 4. Make your change, then re-measure

Either edit the model (add an aggregation, simplify the measure) and re-run the
**same** query, or compare a tuned query variant. Then compute the improvement.

In [ ]:
# Option A (Module 4): re-run the SAME query after you change the model.
# after = benchmark(SLOW_DAX, "After change")

# Option B (Module 8, default): the cheaper query variant.
after = benchmark(TUNED_DAX, "Tuned query")

delta = baseline["Total ms"] - after["Total ms"]
pct   = delta / baseline["Total ms"] if baseline["Total ms"] else 0
print(f"\nTotal change: {delta:>7.0f} ms      Improvement: {pct:6.1%}")

## 5. Record it - one sentence of proof

Write the sentence that proves the gain, for example:
*"The aggregation cut total from 1,800 ms to 240 ms, almost all storage engine."*

Log baseline, after, and the SE/FE split in **metrics-sheet.xlsx**. This same
harness measures any change you make for the rest of the day.

### Optional - DAX Studio
`lab08-benchmark.dax` has the same queries if you want to run them in DAX Studio.
Optional only; the notebook already proves the improvement.